# Alley Kingz -- Free 3D Mesh Batch (open-source, Colab GPU)

Turns AK 2D art into 3D meshes for **$0** on a borrowed cloud GPU. No API key, no paid tool.

**How to run:** `Runtime > Change runtime type > T4 GPU`, then run each cell top to bottom. At the end you download `ak_meshes.zip` -- open the `.obj`/`.glb` in Blender or Unity.

**Model:** TripoSR (VAST-AI, MIT license -- fast, free, roughest). It is the pipeline PROOF. For production-quality PBR (game-ready) swap in **TRELLIS 2** or **Hunyuan3D-2** (last cell) -- same pattern, heavier GPU.

**Input law:** clean, single-subject, front-facing images only. This notebook is prefilled with the 11 boss portraits (clean single characters) as the test batch. For upright hero characters, feed the full-body hero-pose renders once they exist (walk-clip style, not the 4-legged idle).

In [ ]:
# 1. Install TripoSR + marching-cubes (torchmcubes is the usual snag -> built from source here)
!git clone -q https://github.com/VAST-AI-Research/TripoSR.git
%cd TripoSR
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/tatsy/torchmcubes.git
print('install done -- if a build failed, re-run this cell once')

In [ ]:
# 2. Pull clean AK art straight from the live CDN (no upload needed)
import os
os.makedirs('/content/ak_in', exist_ok=True)
BASE='https://alley-kingz.pages.dev/assets'
# 11 boss portraits = clean single-subject test batch. Edit this list for any other assets.
BOSSES=['lot_warden','meter','iron_handler','dock_sovereign','terminus','signal_king',
        'gangrene','marker','cold_saint','regent','the_collar']
for b in BOSSES:
    os.system(f'wget -q -O /content/ak_in/{b}.jpg {BASE}/bosses/{b}.jpg')
print('fetched', len(os.listdir('/content/ak_in')), 'images ->', os.listdir('/content/ak_in'))

In [ ]:
# 2b. Upload YOUR OWN images (e.g. the upright walk $BCARDD from your phone's Download/AK_3D folder).
#     Run this, tap 'Choose Files', pick 0_BCARDD_walk_input.jpg. It joins the batch and gets meshed.
#     (You can skip cell 2 entirely if you only want your own image -- this fills /content/ak_in too.)
import os, shutil
os.makedirs('/content/ak_in', exist_ok=True)
from google.colab import files
up = files.upload()
for fn in up:
    shutil.move(fn, f'/content/ak_in/{fn}')
print('will mesh:', os.listdir('/content/ak_in'))


In [ ]:
# 3. Generate a 3D mesh per image (TripoSR does its own background removal)
import glob
imgs=' '.join(glob.glob('/content/ak_in/*.jpg'))
!python run.py $imgs --output-dir /content/ak_out/ --model-save-format obj
import os
print('mesh folders:', sorted(os.listdir('/content/ak_out')))

In [ ]:
# 4. Zip + download all meshes
!cd /content && zip -qr ak_meshes.zip ak_out
from google.colab import files
files.download('/content/ak_meshes.zip')

## Production upgrade (higher fidelity, game-ready PBR)

TripoSR is the fast/free proof. When you want the real look:

- **TRELLIS 2** (Microsoft, open, commercial-clean) -- outputs PBR (Base/Metallic/Roughness/Opacity), the best for characters + Fortnite-style skins. Run via its HF Space or `git clone microsoft/TRELLIS` on a bigger GPU.
- **Hunyuan3D 2.1** (Tencent) -- high fidelity, has a Gradio demo.
- **Hard-surface** (cars / war-trucks / weapons) -- use a hard-surface-tuned model, generated SEPARATELY from the dog, then assembled on the rig sockets.

**Where to run bigger models:** free tier = Google Colab (T4) / Kaggle (P100, ~30 GPU-h/wk); cheap batch = Vast.ai (~$0.31/hr RTX 4090) or RunPod. A full ~500-asset library batches for a few dollars of compute.

**Next in the pipeline after mesh:** clean bg -> mesh -> Blender retopo/bake for heroes -> rig to ONE shared skeleton (all 106 cards are bipedal, 4 rig families -> one Mixamo/AccuRIG clip set retargets the whole roster) -> attach cosmetics on the drip.js sockets -> import to Unity (`ecosystem/unity_migration/`).